# 2. Please parse large CSV, customers-2000000.csv and keep the memory low.


In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
import zipfile
zipfile.ZipFile("customers-2000000.zip").extractall()

Here we will try to parse and do a monthly subscription count

In [4]:
# Read the CSV file in chunks to avoid memory issues, will only read the "Subscription Date" column
reader = pd.read_csv("customers-2000000.csv", chunksize=100_000, usecols=["Subscription Date"])
total_counts = pd.DataFrame()

for chunk in reader:
    df = chunk
    df["Subscription Date"] = pd.to_datetime(
        df["Subscription Date"],
        errors="coerce"
    )

    df["year"] = df["Subscription Date"].dt.year
    df["month"] = df["Subscription Date"].dt.month
    
    monthly_counts = (
        df.groupby(["year", "month"])
        .size()
        .reset_index(name="count")
    )
    
    total_counts = pd.concat([total_counts, monthly_counts], ignore_index=True)
    
final = (
    total_counts.groupby(["year", "month"], as_index=False)["count"]
    .sum()
)

print("Total rows:", final["count"].sum())
print("Months covered:", len(final))
final

Total rows: 2000000
Months covered: 29


,year,month,count
0,2020,1,70264
1,2020,2,65831
2,2020,3,70477
3,2020,4,68109
4,2020,5,70545
5,2020,6,68144
6,2020,7,70365
7,2020,8,70111
8,2020,9,67861
9,2020,10,70780


Checking the memory usage for chunking processes (Here we only use the first chunk)

In [6]:
first_chunk = next(pd.read_csv("customers-2000000.csv", chunksize=100_000))
full_mb = first_chunk.memory_usage(deep=True).sum() / 1024**2

print(f"One chunk, all 12 columns: {full_mb:.1f} MB")
print(f"Full file loaded at once (estimate): {full_mb * 20:.0f} MB")

One chunk, all 12 columns: 75.3 MB
Full file loaded at once (estimate): 1506 MB


In [9]:
selected_column_first_chunk = next(pd.read_csv("customers-2000000.csv", chunksize=100_000, usecols=["Subscription Date"]))
slim = selected_column_first_chunk.memory_usage(deep=True).sum() / 1024**2

print(f"One chunk, 1 column:       {slim:.1f} MB")

One chunk, 1 column:       6.4 MB


Chunking system enables us to process the large CSV by chunking/batching them into smaller data range.
Every Chunk will be processed using a for loop one at a time and will be released, so it does not use much memory. 